# 01 · Bronze Ingestion
### Fetches raw data from Open-Meteo API and loads into Bronze Delta tables
- Reads watermark per city to know what dates to fetch
- Appends raw data to `weather_hourly_raw` and `air_quality_hourly_raw`
- Updates watermark after every successful city load

In [0]:
%run ../config/pipeline_config

In [0]:
%run ../utils/common_functions

In [0]:
# ── Seed Locations Table ─────────────────────────────────────────────
from pyspark.sql import Row

locations =[
    Row(location_id=1, city="Mumbai", country="India", latitude=19.0760, longitude=72.8777, timezone="Asia/Kolkata", climate_zone="tropical"),
    Row(location_id=2,  city="Delhi",     country="India",          latitude=28.6139,  longitude=77.2090,  timezone="Asia/Kolkata",      climate_zone="Semi-Arid"),
    Row(location_id=3,  city="New York",  country="United States",  latitude=40.7128,  longitude=-74.0060, timezone="America/New_York",  climate_zone="Humid Continental"),
    Row(location_id=4,  city="London",    country="United Kingdom", latitude=51.5074,  longitude=-0.1278,  timezone="Europe/London",     climate_zone="Oceanic"),
    Row(location_id=5,  city="Tokyo",     country="Japan",          latitude=35.6762,  longitude=139.6503, timezone="Asia/Tokyo",        climate_zone="Humid Subtropical"),
    Row(location_id=6,  city="Dubai",     country="UAE",            latitude=25.2048,  longitude=55.2708,  timezone="Asia/Dubai",        climate_zone="Desert"),
    Row(location_id=7,  city="Sydney",    country="Australia",      latitude=-33.8688, longitude=151.2093, timezone="Australia/Sydney",  climate_zone="Oceanic"),
    Row(location_id=8,  city="Sao Paulo", country="Brazil",         latitude=-23.5505, longitude=-46.6333, timezone="America/Sao_Paulo", climate_zone="Tropical"),
    Row(location_id=9,  city="Berlin",    country="Germany",        latitude=52.5200,  longitude=13.4050,  timezone="Europe/Berlin",     climate_zone="Oceanic"),
    Row(location_id=10, city="Nairobi",   country="Kenya",          latitude=-1.2921,  longitude=36.8219,  timezone="Africa/Nairobi",    climate_zone="Tropical Highland"),
]


df_locations = spark.createDataFrame(locations)

(
 df_locations.write
     .format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(TBL_LOCATIONS_RAW)         


)

log("BRONZE", f"Locations table seeded with {df_locations.count()} cities ✅")
display(spark.table(TBL_LOCATIONS_RAW))



In [0]:
# ── Fetch Weather Data from Open-Meteo API ───────────────────────────

def fetch_weather(latitude, longitude, start_date, end_date, timezone):
    params = {
        "latitude"  : latitude,
        "longitude" : longitude,
        "start_date": start_date,
        "end_date"  : end_date,
        "timezone"  : timezone,
        "hourly"    : ",".join(WEATHER_PARAMS)
    }

    response = requests.get(WEATHER_API_URL, params=params)

    if not response.ok:
        raise Exception(f"Weather API failed: {response.status_code} → {response.text}")

    data   = response.json()
    hourly = data["hourly"]
    times  = hourly["time"]
    n      = len(times)

    rows = []
    for i in range(n):
        rows.append({
            "observation_ts"    : times[i],
            "temperature_2m"    : hourly.get("temperature_2m",       [None]*n)[i],
            "relative_humidity" : hourly.get("relative_humidity_2m", [None]*n)[i],
            "precipitation"     : hourly.get("precipitation",         [None]*n)[i],
            "wind_speed_10m"    : hourly.get("wind_speed_10m",        [None]*n)[i],
            "wind_direction_10m": hourly.get("wind_direction_10m",    [None]*n)[i],
            "surface_pressure"  : hourly.get("surface_pressure",      [None]*n)[i],
            "cloud_cover"       : hourly.get("cloud_cover",           [None]*n)[i],
            "weather_code"      : hourly.get("weather_code",          [None]*n)[i],
        })

    log("WEATHER_API", f"Fetched {len(rows)} hourly records | {start_date} → {end_date}")
    return rows, response.url

In [0]:
# ── Fetch Air Quality Data from Open-Meteo API ───────────────────────
def fetch_air_quality(latitude,longitude,start_date,end_date,timezone):
    params = {
        "latitude"  : latitude,
        "longitude" : longitude,
        "start_date": start_date,
        "end_date"  : end_date,
        "timezone"  : timezone,
        "hourly"    : ",".join(AIR_QUALITY_PARAMS)
    }
    response = requests.get(AIR_QUALITY_API_URL, params=params)
   
    if response.status_code != 200:
        raise Exception(f"Air Quality API failed: {response.status_code} → {response.text}")

    data = response.json()
    hourly = data["hourly"]
    times = hourly['time']
    n = len(times)

    rows =[]
    for i in range(n):
        rows.append({
            "observation_ts"  : times[i],
            "pm2_5"           : hourly.get("pm2_5",             [None]*n)[i],
            "pm10"            : hourly.get("pm10",              [None]*n)[i],
            "carbon_monoxide" : hourly.get("carbon_monoxide",   [None]*n)[i],
            "nitrogen_dioxide": hourly.get("nitrogen_dioxide",  [None]*n)[i],
            "ozone"           : hourly.get("ozone",             [None]*n)[i],
            "dust"            : hourly.get("dust",              [None]*n)[i],
            "european_aqi"    : hourly.get("european_aqi",      [None]*n)[i],
            "us_aqi"          : hourly.get("us_aqi",            [None]*n)[i],
        })

    log("AQ_API", f"Fetched {len(rows)} hourly records | {start_date} → {end_date}")
    return rows, response.url

In [0]:
from datetime import datetime
import uuid

run_id    = str(uuid.uuid4())
run_ts    = datetime.now()
locations = spark.table(TBL_LOCATIONS_RAW).collect()

log("BRONZE", f"Starting ingestion run | run_id={run_id}")
log("BRONZE", f"Total cities to process: {len(locations)}")

for loc in locations:
    location_id = loc["location_id"]
    city        = loc["city"]
    country     = loc["country"]
    latitude    = loc["latitude"]
    longitude   = loc["longitude"]
    timezone    = loc["timezone"]

    log("BRONZE", f"Processing city: {city} ({country})")

    # ── Weather ──────────────────────────────────────────────────────
    try:
        start_date, end_date = get_load_date_range("weather_hourly", location_id)

        rows, source_url = fetch_weather(
            latitude, longitude, start_date, end_date, timezone
        )

        df_weather = spark.createDataFrame(rows) \
            .withColumn("location_id",        F.lit(location_id).cast("int")) \
            .withColumn("city",               F.lit(city)) \
            .withColumn("observation_ts",     F.to_timestamp("observation_ts")) \
            .withColumn("temperature_2m",     F.col("temperature_2m").cast("double")) \
            .withColumn("relative_humidity",  F.col("relative_humidity").cast("double")) \
            .withColumn("precipitation",      F.col("precipitation").cast("double")) \
            .withColumn("wind_speed_10m",     F.col("wind_speed_10m").cast("double")) \
            .withColumn("wind_direction_10m", F.col("wind_direction_10m").cast("double")) \
            .withColumn("surface_pressure",   F.col("surface_pressure").cast("double")) \
            .withColumn("cloud_cover",        F.col("cloud_cover").cast("double")) \
            .withColumn("weather_code",       F.col("weather_code").cast("int")) \
            .withColumn("_ingested_at",       F.lit(run_ts)) \
            .withColumn("_source_url",        F.lit(str(source_url))) \
            .withColumn("_batch_date",        F.lit(end_date).cast("date"))

        df_weather.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(TBL_WEATHER_RAW)

        update_watermark("weather_hourly", location_id, end_date)
        log("BRONZE", f"✅ Weather done for {city} | rows={len(rows)}")

    except Exception as e:
        log("BRONZE", f"❌ Weather failed for {city} | error={str(e)}", status="ERROR")

    # ── Air Quality ──────────────────────────────────────────────────
    try:
        start_date, end_date = get_load_date_range("air_quality_hourly", location_id)

        rows, source_url = fetch_air_quality(
            latitude, longitude, start_date, end_date, timezone
        )

        df_aq = spark.createDataFrame(rows) \
            .withColumn("location_id",        F.lit(location_id).cast("int")) \
            .withColumn("city",               F.lit(city)) \
            .withColumn("observation_ts",     F.to_timestamp("observation_ts")) \
            .withColumn("pm2_5",              F.col("pm2_5").cast("double")) \
            .withColumn("pm10",               F.col("pm10").cast("double")) \
            .withColumn("carbon_monoxide",    F.col("carbon_monoxide").cast("double")) \
            .withColumn("nitrogen_dioxide",   F.col("nitrogen_dioxide").cast("double")) \
            .withColumn("ozone",              F.col("ozone").cast("double")) \
            .withColumn("dust",               F.col("dust").cast("double")) \
            .withColumn("european_aqi",       F.col("european_aqi").cast("int")) \
            .withColumn("us_aqi",             F.col("us_aqi").cast("int")) \
            .withColumn("_ingested_at",       F.lit(run_ts)) \
            .withColumn("_source_url",        F.lit(str(source_url))) \
            .withColumn("_batch_date",        F.lit(end_date).cast("date"))

        df_aq.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(TBL_AIR_QUALITY_RAW)

        update_watermark("air_quality_hourly", location_id, end_date)
        log("BRONZE", f"✅ Air quality done for {city} | rows={len(rows)}")

    except Exception as e:
        log("BRONZE", f"❌ Air quality failed for {city} | error={str(e)}", status="ERROR")

log("BRONZE", f"✅ Ingestion run complete | run_id={run_id}")